# Registering your own optimizer

`l2co` ships with 30+ built-in optimizers from [`optax`](https://github.com/google-deepmind/optax) and [`evosax`](https://github.com/RobertTLange/evosax), all referenced by string name through an [`OptimizationStep`](api.md/#l2co.OptimizationStep). This guide shows how to **port your own optimizer** so it can be referenced the same way and driven by the [`RolloutWrapper`](api.md/#l2co.RolloutWrapper) (and the higher-level [`RunOptimization`](api.md/#l2co.RunOptimization) pipeline) exactly like a built-in one.

There are three steps:

1. Write a **factory** that returns an [`UpdateClass`](api.md/#l2co.UpdateClass).
2. **Register** it under a name with [`register_optimizer`](api.md/#l2co.register_optimizer).
3. Reference that name from an [`OptimizationStep`](api.md/#l2co.OptimizationStep).

This is the same extension point that the sibling [`rl2co`](https://github.com/bessagroup/rl2co) package uses to inject its reinforcement-learning meta-optimizer into `l2co` without `l2co` ever depending on `rl2co`.

## How optimizers are resolved

Every optimizer name lives in a registry: a plain dictionary mapping `name -> factory`. When you build a [`RolloutWrapper`](api.md/#l2co.RolloutWrapper), [`RunState.init`](api.md/#l2co.RunState) looks the name up via [`optimizer_mapping`](api.md/#l2co.optimizer_mapping) and calls the factory to obtain an [`UpdateClass`](api.md/#l2co.UpdateClass):

```text
OptimizationStep(optimizer='myopt', ...)
        |
        v  optimizer_mapping('myopt')
   factory(**hyperparameters, task, opt_hash, bounded, stop_fn)
        |
        v
   UpdateClass(init_fn, step_fn, popsize, hash, stop_fn)
        |
        v  consumed by the rollout machinery (init_ / step / run_)
   loss trajectories (HistoryState / OptHistory)
```

### The factory contract

`RunState.init` always calls your factory with this keyword contract:

- **`task`** ([`Task`](https://github.com/bessagroup/l2co-tasks)) -- provides `task.model` (the parameters to optimize), `task.loss_fn`, and `task.pass_rng`.
- **`opt_hash`** (`int`) -- a stable id stamped into every [`OptHistory`](api.md/#l2co.OptHistory) entry (`update_step`) so the trajectory can be traced back to this optimizer configuration.
- **`bounded`** (`tuple[float | None, float | None]`) -- box constraints applied to the parameters after each step.
- **`stop_fn`** (`Callable`) -- a stopping function derived from `OptimizationStep.stopping_criteria`; forward it through to the `UpdateClass` unchanged.
- **`**hyperparameters`** -- everything in `OptimizationStep.hyperparameters` (e.g. `learning_rate`). These are *your* optimizer's knobs.

The factory must return an [`UpdateClass`](api.md/#l2co.UpdateClass).

### The `UpdateClass` contract

[`UpdateClass`](api.md/#l2co.UpdateClass) is a static (JIT-friendly) `eqx.Module`. The five fields every
optimizer must supply:

| Field | Meaning |
| --- | --- |
| `init_fn` | Builds the initial optimizer state. Signature: `init_fn(params, key=None, **sample) -> opt_state`. |
| `step_fn` | Performs **one** optimization step (see below). |
| `popsize` | Number of candidates evaluated per step (`1` for gradient-based optimizers). |
| `hash` | The `opt_hash` passed to the factory. |
| `stop_fn` | The `stop_fn` passed to the factory (or the never-stop default). |

and five optional ones, all defaulted, which let the optimizer take part in
budget accounting and in the state transfer applied when a meta-optimizer
switches away from it and back:

| Field | Meaning |
| --- | --- |
| `max_fevals_per_step` | Worst-case evaluations one step can bill, when that is more than `popsize` (line-search optimizers). Read it through `fevals_bound`. |
| `family` | `'gradient'` (default), `'population'` or `'distribution'` — see below. |
| `ask_fn` | `ask_fn(opt_state, key) -> (params, opt_state)`; the optimizer's own sampling step. Required for `'distribution'`, ignored otherwise. |
| `transfer_read_fn` | `transfer_read_fn(opt_state) -> (sigma, conf)`; projects your state onto the scale a switch carries. |
| `transfer_write_fn` | `transfer_write_fn(bundle, opt_state) -> opt_state`; folds a carried scale back into your state. |

**`step_fn` is where the work happens.** The rollout scan calls it as:

```python
(new_params, new_opt_state, new_key), history = step_fn((params, opt_state, key), sample=sample)
```

- The **carry** is the 3-tuple `(params, opt_state, key)`. `params` always carries a leading `popsize` axis, i.e. shape `(popsize, *param_shape)` on every array leaf.
- `sample` is a batch of the task dataset (an empty dict for dataset-free tasks). Combine the parameters back into a model with `eqx.combine(params, static)` before evaluating the loss.
- It must return the **new carry** plus an [`OptHistory`](api.md/#l2co.OptHistory) record for this step.

An [`OptHistory`](api.md/#l2co.OptHistory) entry stores, per step:

- `loss` -- shape `(popsize,)`; the loss of the *current* population (before the update).
- `params`, `grads` -- the current population's parameters and gradients (use `jnp.nan` gradients for derivative-free methods).
- `update_step` -- set to `opt_hash`.
- `fevals` -- number of function evaluations this step (`1` for a single gradient step, `popsize` for a population sweep).
- `iterations` -- number of inner iterations this step (usually `1`).

Box constraints are *your* responsibility: clip the new parameters to `bounded` before returning them, mirroring the built-in factories in `l2co._src.optimizers.update_class`.

In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import jax.random as jr

from l2co import OptimizationStep, RolloutWrapper, UpdateClass, tree_utils
from l2co.history import OptHistory
from l2co.optimizers import optimizer_mapping, optimizers, register_optimizer
from l2co.sampling import normal_sampling
from l2co_tasks import create_bbob_task

## Writing a custom optimizer

As a worked example we implement **heavy-ball (momentum) gradient descent** from scratch -- not via `optax`, so the full contract is visible. The optimizer state is a velocity buffer; each step computes the loss and gradient of the current population, updates the velocity, and takes a step.

We use [`tree_utils.vmapped_loss_and_grad`](api.md/#l2co.tree_utils) to evaluate loss and gradients across the leading `popsize` axis, exactly as the built-in `optax` factory does.

In [ ]:
def heavyball_factory(
    task,
    opt_hash,
    bounded=(None, None),
    stop_fn=None,
    learning_rate=1e-2,
    momentum=0.9,
    **kwargs,
):
    """Factory for a hand-rolled heavy-ball (momentum) optimizer.

    Matches the registry contract called by ``RunState.init``:
    ``task``, ``opt_hash``, ``bounded`` and ``stop_fn`` are supplied by
    the rollout machinery; ``learning_rate`` and ``momentum`` arrive
    through ``OptimizationStep.hyperparameters``.
    """
    # Static (non-array) part of the model; recombined with the live
    # parameters inside the step via ``eqx.combine``.
    _, static = eqx.partition(task.model, eqx.is_inexact_array)
    loss_fn = task.loss_fn

    def init_fn(params, key=None, **sample):
        # One velocity buffer per leaf, shaped like params (popsize, dim).
        return jax.tree.map(jnp.zeros_like, params)

    def step_fn(carry, sample, *args, **kwargs):
        params, velocity, key = carry

        # Loss + gradient of the current population (leading popsize axis).
        loss, grads = tree_utils.vmapped_loss_and_grad(
            eqx.combine(params, static), loss_fn, sample
        )

        # Heavy-ball update.
        velocity = jax.tree.map(lambda v, g: momentum * v + g, velocity, grads)
        new_params = jax.tree.map(
            lambda p, v: p - learning_rate * v, params, velocity
        )

        # Apply box constraints.
        new_params = jax.tree.map(lambda p: jnp.clip(p, *bounded), new_params)

        history = OptHistory(
            loss=loss,
            params=params,
            grads=grads,
            update_step=jnp.array(opt_hash, dtype=int),
            fevals=jnp.array(1, dtype=int),
            iterations=jnp.array(1, dtype=int),
        )

        new_key, _ = jr.split(key)
        return (new_params, velocity, new_key), history

    return UpdateClass(
        init_fn=init_fn,
        step_fn=step_fn,
        popsize=1,  # gradient-based: a single point
        hash=opt_hash,
        stop_fn=stop_fn,
    )

## Registering the optimizer

[`register_optimizer`](api.md/#l2co.register_optimizer) inserts the factory into the global registry under a (lowercased) name. After this call, `'heavyball'` resolves through [`optimizer_mapping`](api.md/#l2co.optimizer_mapping) like any built-in optimizer.

In a package, you would call `register_optimizer` at import time so that simply importing your package wires the optimizer into `l2co`.

In [ ]:
register_optimizer("heavyball", heavyball_factory)

# It is now part of the registry and resolvable by name.
print("heavyball" in optimizers)
optimizer_mapping("heavyball")

## Using it with the `RolloutWrapper`

From here on, the custom optimizer behaves like any built-in one. Build an [`OptimizationStep`](api.md/#l2co.OptimizationStep) that references it by name, passing the optimizer-specific knobs through `hyperparameters`, then hand it to [`RolloutWrapper.init`](api.md/#l2co.RolloutWrapper.init).

In [ ]:
task = create_bbob_task(fn_name="sphere", seed=42, dimensionality=10)

heavyball_step = OptimizationStep(
    optimizer="heavyball",
    hyperparameters={"learning_rate": 5e-2, "momentum": 0.9},
)

rollout = RolloutWrapper.init(
    optimizer=heavyball_step,
    task=task,
    sampler=normal_sampling,
    key=jr.key(0),
    bounded=(-5.0, 5.0),
    verbose=False,
)

run_state, batch_state, history_state = rollout.batch_evaluate(
    key=jr.key(123), n_iterations=200, n_realizations=10
)

print(f"best loss found: {float(jnp.min(run_state.best_loss)):.4g}")

The returned [`HistoryState`](api.md/#l2co.HistoryState) records the optimization trajectory, just like for a built-in optimizer, and can be converted to an `xarray.Dataset` for analysis or plotting.

In [ ]:
import matplotlib.pyplot as plt

xarr = history_state.to_xarray_realizations()

# Best-so-far loss, averaged over realizations.
traj = xarr["output_min"].mean("realization").squeeze()

fig, ax = plt.subplots()
ax.plot(traj)
ax.set_yscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel("loss (mean over realizations)")
ax.set_title("heavyball on BBOB sphere (10D)")
fig.tight_layout()

## Population-based optimizers

The same contract covers derivative-free / population-based optimizers -- only a few details change:

- Set `popsize > 1`. `RunState.init` will sample a population of that size, so `params` (and therefore `loss`) carry a leading axis of length `popsize`.
- Evaluate the whole population at once with [`tree_utils.vmapped_loss`](api.md/#l2co.tree_utils) (no gradients needed), and store `grads` as `jnp.nan` of the right shape in the [`OptHistory`](api.md/#l2co.OptHistory).
- Set `fevals` to `popsize` (one evaluation per population member).

For a fully worked example, see the EvoSax factories in `l2co._src.optimizers.update_class` (`evosax_distribution_based_fn` / `evosax_population_based_fn`), and `l2co._src.optimizers.random_search` for the one-shot random-search variant.

## Taking part in state transfer

Everything above is enough to *run* your optimizer. One more step lets it
take part in a **switch**: when a meta-optimizer moves from another
optimizer to yours, `l2co` can hand over the scale of the search so far,
instead of your optimizer restarting at whatever its constructor chose.
On a menu whose members' initial scales differ by three orders of
magnitude, that gap is the difference between resuming a converged search
and starting over.

Declare it with `build_transfer_fns`, which returns the reader/writer
pair for your optimizer's family:

```python
from l2co.optimizers import FAMILY_GRADIENT, build_transfer_fns

read_fn, write_fn = build_transfer_fns(
    'heavyball', FAMILY_GRADIENT, hyperparameters
)

return UpdateClass(
    init_fn=init_fn,
    step_fn=step_fn,
    popsize=1,
    hash=opt_hash,
    stop_fn=stop_fn,
    family=FAMILY_GRADIENT,
    transfer_read_fn=read_fn,
    transfer_write_fn=write_fn,
)
```

Pass `hyperparameters` through: the pair is built at construction time
precisely because it needs values your state does not retain — the
learning rate, any clip bounds, and (for evosax algorithms) the ravel
function used to flatten a solution pytree.

Which `family` you declare decides what a switching caller does with the
population, so it is not cosmetic:

- **`'gradient'`** — unit population. The caller hands you the best point
  so far. Heavy-ball is this case.
- **`'population'`** — your stored population *is* your state; the caller
  hands you a population with the best point injected.
- **`'distribution'`** — you hold an explicit search distribution. The
  caller writes the location and scale into your state and then calls your
  `ask_fn` so **you** draw the population. If your `step_fn` caches
  anything from its own sampling (noise draws, perturbations) and reads it
  back on the next update, you *must* be this family and you *must*
  supply `ask_fn` — otherwise a caller substitutes a population your
  cached draws do not correspond to, and your adaptation silently
  proceeds against a fiction. See `docs/adr/0012`.

Declare `family` explicitly rather than letting it default; the default
is `'gradient'` and would misdescribe anything else.

If your optimizer holds no notion of scale, pass nothing: the defaults
report "no information" and every writer skips.


## Summary

To port your own optimizer so it works with the [`RolloutWrapper`](api.md/#l2co.RolloutWrapper):

1. **Write a factory** `f(task, opt_hash, bounded, stop_fn, **hyperparameters) -> UpdateClass`.
2. Inside it, define `init_fn` (build the optimizer state) and `step_fn` (one step: evaluate -> update -> emit an [`OptHistory`](api.md/#l2co.OptHistory), clipping to `bounded`).
3. **Declare `family`** and, if it holds a scale, wire
   `transfer_read_fn` / `transfer_write_fn` via `build_transfer_fns` —
   plus `ask_fn` if the family is `'distribution'`.
4. **Register** it with `register_optimizer('myopt', f)` -- at import time if you ship it in a package.
5. Reference it by name from an [`OptimizationStep`](api.md/#l2co.OptimizationStep), passing knobs through `hyperparameters`.

Because the optimizer is now in the registry, it also works with the higher-level [`RunOptimization`](api.md/#l2co.RunOptimization) pipeline and the full `l2co` experiment tooling -- no further wiring required.